# Project 1
## Gender Prediction

In [1]:
import json
from collections import Counter
from string import punctuation, digits

import pandas as pd
import numpy as np
from faker import Faker
from names_dataset import NameDataset

### Create fake names

In [2]:
fake_gen = Faker()

In [3]:
def generate_name(prob: float = 0.5) -> str:
    """
    Generate and return random names. Male and female names are split based on `prob`

    Args:
        prob (float): Probability of female name generation
    """

    if np.random.rand() > prob:
        return fake_gen.name_female()
    return fake_gen.name_male()

### Create a `DataFrame` with fake names

In [4]:
df = pd.DataFrame({"Name": [generate_name() for _ in range(500)]})

In [5]:
df

,Name
0,Alexis Jones
1,Peter Hobbs
2,Joshua Cortez
3,Donna Fischer
4,Jared Moore
...,...
495,Alexis Palmer
496,William Martinez
497,Melissa Martinez
498,Steven Griffith


### Extract first name and last name

In [6]:
name_suffixes = {"MD", "PhD", "MBA", "DVM", "DDS", "III", "Jr.", "Sr.", "V", "IV", "II"}

In [7]:
name_prefixes = {"Dr.", "Mr.", "Mrs.", "Ms."}

In [8]:
def extract_fname(full_name: str) -> str:
    """
    Extract the first name from the given full name.
    """

    split_txt = full_name.split()
    if len(split_txt) > 2:
        if split_txt[0] in name_prefixes:
            return split_txt[1]
    return split_txt[0]

In [9]:
def extract_lname(full_name: str) -> str:
    """
    Extract the first name from the given full name.
    """
    split_txt = full_name.split()
    if len(split_txt) == 4:
        return split_txt[2]
    if len(split_txt) == 3:
        if split_txt[-1] in name_suffixes:
            return split_txt[1]
    return split_txt[-1]

In [10]:
df["First Name"] = df["Name"].apply(extract_fname)

In [11]:
df["Last Name"] = df["Name"].apply(extract_lname)

In [12]:
df

,Name,First Name,Last Name
0,Alexis Jones,Alexis,Jones
1,Peter Hobbs,Peter,Hobbs
2,Joshua Cortez,Joshua,Cortez
3,Donna Fischer,Donna,Fischer
4,Jared Moore,Jared,Moore
...,...,...,...
495,Alexis Palmer,Alexis,Palmer
496,William Martinez,William,Martinez
497,Melissa Martinez,Melissa,Martinez
498,Steven Griffith,Steven,Griffith


### Predict gender

In [13]:
nd = NameDataset()

In [14]:
def name_to_gender(name: str) -> str | None:
    """
    Predict the gender of the given name using `NameDataset` from `names_dataset` library.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["first_name"]
    if query is None:
        return None
    return max(query["gender"], key=query["gender"].get)

In [15]:
def gender_prob(name):
    """
    Return the probability of the given name being male or female.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["first_name"]
    if query is None:
        return None
    return max(query["gender"].values())

In [16]:
df["Gender"] = df["First Name"].apply(name_to_gender)

In [17]:
df["Gender Probability"] = df["First Name"].apply(gender_prob)

In [18]:
df

,Name,First Name,Last Name,Gender,Gender Probability
0,Alexis Jones,Alexis,Jones,Male,0.859
1,Peter Hobbs,Peter,Hobbs,Male,0.988
2,Joshua Cortez,Joshua,Cortez,Male,0.987
3,Donna Fischer,Donna,Fischer,Female,0.992
4,Jared Moore,Jared,Moore,Male,0.982
...,...,...,...,...,...
495,Alexis Palmer,Alexis,Palmer,Male,0.859
496,William Martinez,William,Martinez,Male,0.990
497,Melissa Martinez,Melissa,Martinez,Female,0.992
498,Steven Griffith,Steven,Griffith,Male,0.992


### Predict Country

In [19]:
def name_to_country(name: str) -> str | None:
    """
    Predict the country of origin based on the given name.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["last_name"]
    if query is None:
        return None
    return max(query["country"], key=query["country"].get)

In [20]:
def country_prob(name: str) -> float | None:
    """
    Return the probability of the given name belonging to the predicted country.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["last_name"]
    if query is None:
        return None
    return max(query["country"].values())

I used last name as the indicator of nationality because it is more stable and predictable.

In [21]:
df["Country"] = df["Last Name"].apply(name_to_country)

In [22]:
df["Country Probability"] = df["Last Name"].apply(country_prob)

In [23]:
df

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
0,Alexis Jones,Alexis,Jones,Male,0.859,United States,0.492
1,Peter Hobbs,Peter,Hobbs,Male,0.988,United Kingdom,0.468
2,Joshua Cortez,Joshua,Cortez,Male,0.987,United States,0.355
3,Donna Fischer,Donna,Fischer,Female,0.992,Germany,0.520
4,Jared Moore,Jared,Moore,Male,0.982,United States,0.583
...,...,...,...,...,...,...,...
495,Alexis Palmer,Alexis,Palmer,Male,0.859,United States,0.406
496,William Martinez,William,Martinez,Male,0.990,United States,0.304
497,Melissa Martinez,Melissa,Martinez,Female,0.992,United States,0.304
498,Steven Griffith,Steven,Griffith,Male,0.992,United States,0.691


In [24]:
df[["Country Probability", "Gender Probability"]] = df[
    ["Country Probability", "Gender Probability"]
].astype(np.float32)

In [ ]:
# names with prefix and/or suffixes
df[df["Name"].str.split().apply(len) > 2]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
32,Samantha Mcconnell MD,Samantha,Mcconnell,Female,0.981,United States,0.445
34,Monica Cochran DVM,Monica,Cochran,Female,0.992,United States,0.908
43,Eric Brown Jr.,Eric,Brown,Male,0.992,United States,0.515
50,James Morton Jr.,James,Morton,Male,0.982,United Kingdom,0.464
65,Mr. Christopher Bryan,Christopher,Bryan,Male,0.991,United States,0.396
104,Mr. Vincent Hensley,Vincent,Hensley,Male,0.990,United States,0.915
114,Sandra Sullivan PhD,Sandra,Sullivan,Female,0.991,United States,0.620
129,Dr. Charles Jenkins,Charles,Jenkins,Male,0.984,United States,0.557
141,Renee Graham PhD,Renee,Graham,Female,0.951,United Kingdom,0.410
162,Lindsey Wheeler DVM,Lindsey,Wheeler,Female,0.975,United States,0.537


Names with prefixes and/or suffixes are correctly processed

In [26]:
df["Country"].value_counts()

Country
United States     390
United Kingdom     52
Colombia           19
Germany             8
Ireland             7
Mexico              6
France              5
Brazil              4
Malaysia            3
Hong Kong           2
Denmark             2
Chile               1
Netherlands         1
Name: count, dtype: int64

In [27]:
df[df["Country"] == "Germany"]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
3,Donna Fischer,Donna,Fischer,Female,0.992,Germany,0.520
82,Lucas Weber,Lucas,Weber,Male,0.987,Germany,0.413
211,Lawrence Schmidt,Lawrence,Schmidt,Male,0.973,Germany,0.633
237,Bryan Schmidt,Bryan,Schmidt,Male,0.992,Germany,0.633
240,Catherine Wagner,Catherine,Wagner,Female,0.994,Germany,0.404
304,Anthony Meyer,Anthony,Meyer,Male,0.990,Germany,0.292
460,Stacey Fischer,Stacey,Fischer,Female,0.957,Germany,0.520
469,Robin Schmidt,Robin,Schmidt,Male,0.776,Germany,0.633


In [28]:
df[df["Country"].isin(["Denmark", "France"])]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
275,Jerry Jensen,Jerry,Jensen,Male,0.969,Denmark,0.565
284,Cassidy Vincent,Cassidy,Vincent,Female,0.831,France,0.382
293,Christopher Lambert,Christopher,Lambert,Male,0.991,France,0.433
373,Christopher Jensen,Christopher,Jensen,Male,0.991,Denmark,0.565
388,Charles Simon,Charles,Simon,Male,0.984,France,0.348
420,Amy Simon,Amy,Simon,Female,0.970,France,0.348
492,Henry Lambert,Henry,Lambert,Male,0.989,France,0.433


In [29]:
df["Gender"].value_counts()

Gender
Male      276
Female    224
Name: count, dtype: int64

In [30]:
df.to_csv("Data/Gender.csv")

## Bonus
### Pytopia Members' Gender Prediction

Extracted Pytopia's Chat as json file from January of 2024 to June of 2025. I am going to extract the name of the members and predict their gender.

In [31]:
with open("Data/Pytopia-Data-Export.json") as f:
    data = json.load(f)

In [32]:
type(data)

dict

In [33]:
data.keys()

dict_keys(['name', 'type', 'id', 'messages'])

In [34]:
data["messages"][0]

{'id': 65892,
 'type': 'message',
 'date': '2024-01-01T00:05:06',
 'date_unixtime': '1704054906',
 'edited': '2024-01-01T00:12:32',
 'edited_unixtime': '1704055352',
 'from': 'Sky',
 'from_id': 'user1465452745',
 'reply_to_message_id': 65885,
 'text': 'اگر ایران هستی میشه در دسترس داشت (با تغییر منطقه زبانی) در غیر این صورت شایت مایکروسافت داره میتونی دانلود کنی و اموزش نصبش هم در سایت مایکروسافت قرار داده شده',
 'text_entities': [{'type': 'plain',
   'text': 'اگر ایران هستی میشه در دسترس داشت (با تغییر منطقه زبانی) در غیر این صورت شایت مایکروسافت داره میتونی دانلود کنی و اموزش نصبش هم در سایت مایکروسافت قرار داده شده'}],
 'reactions': [{'type': 'emoji',
   'count': 1,
   'emoji': '🙏',
   'recent': [{'from': 'Gorji',
     'from_id': 'user6030521554',
     'date': '2024-01-01T00:12:32'}]}]}

In [35]:
user_names = []
user_id = set()
for message in data["messages"]:
    if message.get("from") is None:
        continue
    if message.get("from_id") in user_id:
        continue
    user_id.add(message["from_id"])
    user_names.append(message.get("from"))

In [36]:
user_names[0:10]

['Sky',
 'Mehdirrrrrr',
 'Raha',
 'Homeira',
 'Khosro',
 'Ali Hejazi',
 'Yeganeh Mhr \U0001fa75',
 'mandana',
 'A',
 'S']

In [37]:
first_names = [name.split()[0].title() for name in user_names]

### Write a function to validate names

In [38]:
invalid_chars = set(punctuation + digits)  # from string module

In [39]:
def validate_name(name: str) -> bool:
    if len(name) < 3:
        return False
    return not any(char in invalid_chars for char in name)

In [40]:
valid_first_names = list(filter(validate_name, first_names))

In [41]:
Counter(valid_first_names).most_common(10)

[('Ali', 77),
 ('Mohammad', 65),
 ('Amir', 56),
 ('Alireza', 41),
 ('Zahra', 33),
 ('Maryam', 30),
 ('Reza', 30),
 ('Fatemeh', 28),
 ('Mahdi', 23),
 ('Mehdi', 19)]

### Creating the `DataFrame`

In [42]:
df = pd.DataFrame({"Name": valid_first_names})

In [43]:
df

,Name
0,Sky
1,Mehdirrrrrr
2,Raha
3,Homeira
4,Khosro
...,...
1981,Danial
1982,Najmeh
1983,Mohammad
1984,Mahan


In [44]:
df["Gender"] = df["Name"].apply(name_to_gender)

In [45]:
df.head(20).dropna()

,Name,Gender
0,Sky,Male
2,Raha,Female
3,Homeira,Female
4,Khosro,Male
5,Ali,Male
6,Yeganeh,Female
7,Mandana,Female
8,Kavosh,Male
9,Ahmadreza,Male
10,Fati,Female


In [46]:
df[df["Name"] == "Arad"]

,Name,Gender
205,Arad,Male
701,Arad,Male


Apparently there is another Arad in the group chat :)

In [47]:
df["Gender"].value_counts()

Gender
Male      1090
Female     642
Name: count, dtype: int64

In [48]:
print(df["Gender"].isna().sum())  # null values count

254
